# BirdBase × AVONET — Data Merging & Feature Engineering

This notebook merges two bird trait databases — **AVONET** (morphological measurements) and **BirdBase** (ecology, diet, elevation, nest type) — into a single combined dataset keyed on `avibase_id`.

**Workflow overview:**
1. Load raw datasets
2. Cast numeric columns (both sources have MISSING strings and mixed types)
3. Harmonise categorical vocabularies so conflicting columns are comparable
4. Resolve BirdBase duplicates (67 species appear twice)
5. Rename columns to clear, readable names
6. Outer-merge on `avibase_id` and add provenance flags

## 1 · Setup

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
import warnings

# ── Display settings ──────────────────────────────────────────────────────────
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1200)
pd.set_option("display.max_colwidth", 100)

# Suppress mixed-type warnings that arise from messy raw data
warnings.filterwarnings("ignore")

## 2 · Load Raw Data

In [ ]:
birdbase = pd.read_csv("birdbase_uncleaned_final_dataset.csv")
avonet   = pd.read_csv("avonet_uncleaned_final_dataset.csv")

print(f"AVONET   : {avonet.shape[0]:,} rows × {avonet.shape[1]} cols")
print(f"BirdBase : {birdbase.shape[0]:,} rows × {birdbase.shape[1]} cols")

In [ ]:
avonet.head()

In [ ]:
birdbase.head()

## 3 · AVONET — Numeric Type Casting

All morphometric and geographic columns were read as `object` because some rows contain
commas or whitespace.  We strip those and coerce to `float64`; anything unparseable becomes `NaN`.

In [ ]:
AVONET_NUMERIC_COLS = [
    'beak_culmen_avg', 'beak_culmen_avg_m', 'beak_culmen_avg_f',
    'beak_nares_avg',  'beak_nares_avg_m',  'beak_nares_avg_f',
    'beak_width_avg',  'beak_width_avg_m',  'beak_width_avg_f',
    'beak_depth_avg',  'beak_depth_avg_m',  'beak_depth_avg_f',
    'tarsus_avg',      'tarsus_avg_m',      'tarsus_avg_f',
    'wing_len_avg',    'wing_len_avg_m',    'wing_len_avg_f',
    'kipps_avg',       'kipps_avg_m',       'kipps_avg_f',
    'secondary_avg',   'secondary_avg_m',   'secondary_avg_f',
    'hwi_avg',         'hwi_avg_m',         'hwi_avg_f',
    'tail_avg',        'tail_avg_m',        'tail_avg_f',
    'total_individuals', 'female_count', 'male_count',
    'mass_avg', 'habitat_density', 'migration',
    'lat_min', 'lat_max', 'lat_centroid', 'lon_centroid', 'range_size',
]

for col in AVONET_NUMERIC_COLS:
    avonet[col] = pd.to_numeric(
        avonet[col].astype(str).str.replace(",", "").str.strip(),
        errors="coerce"
    )

avonet.info()

## 4 · BirdBase — Cleaning & Feature Engineering

BirdBase stores missing values as the string `"MISSING"` rather than `NaN`.
We normalise these first, then work through each feature group.

### 4.1 · Body Mass

Both datasets record body mass in grams.  BirdBase `mass_avg` is stored as `object`
due to the `"MISSING"` sentinel — we cast it to numeric.

In [ ]:
birdbase["mass_avg"] = pd.to_numeric(
    birdbase["mass_avg"].astype(str).str.replace(",", "").str.strip(),
    errors="coerce"
)

print(f"dtype  : {birdbase['mass_avg'].dtype}")
print(f"nulls  : {birdbase['mass_avg'].isna().sum():,}")
print()
print(birdbase["mass_avg"].describe())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(birdbase["mass_avg"].dropna(), bins=50, kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("Distribution of mass_avg (Raw)")
axes[0].set_xlabel("Mass (g)")
axes[0].set_ylabel("Count")

sns.histplot(birdbase["mass_avg"].dropna(), bins=50, kde=True, ax=axes[1], color="coral", log_scale=True)
axes[1].set_title("Distribution of mass_avg (Log Scale)")
axes[1].set_xlabel("Mass (g) — log scale")
axes[1].set_ylabel("Count")

plt.suptitle("Bird Body Mass Distribution (BirdBase)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

Body mass is strongly right-skewed (a few very heavy species like ostriches pull the mean far above the median).  Log-scale is the appropriate axis for downstream analysis.

### 4.2 · Primary Diet — Vocabulary Harmonisation

BirdBase uses fine-grained diet labels (e.g. `Invertebrate`, `Fish`, `Fruit`) while AVONET uses
broad trophic levels (`Carnivore`, `Herbivore`, `Omnivore`, `Scavenger`).
We map BirdBase's labels to the same four-class schema so the two sources can be compared.

In [ ]:
print("BirdBase diet_primary values:")
print(birdbase["diet_primary"].value_counts())
print()
print("AVONET trophic_level values:")
print(avonet["trophic_level"].value_counts())

In [ ]:
diet_mapping = {
    # → Carnivore
    "Invertebrate": "Carnivore",
    "Vertebrate":   "Carnivore",
    "Fish":         "Carnivore",
    "Carnivore":    "Carnivore",
    "Ovivore":      "Carnivore",
    # → Herbivore
    "Fruit":        "Herbivore",
    "Seed":         "Herbivore",
    "Nectar":       "Herbivore",
    "Herbivore":    "Herbivore",
    "Plant":        "Herbivore",
    "Beeswax":      "Herbivore",
    # → Omnivore
    "Omnivore":     "Omnivore",
    # → Scavenger
    "Scavenger":    "Scavenger",
    # → MISSING
    "No Information": "MISSING",
}

birdbase["diet_primary"] = birdbase["diet_primary"].map(diet_mapping).fillna("MISSING")

print("BirdBase diet_primary after harmonisation:")
print(birdbase["diet_primary"].value_counts())

### 4.3 · Trophic Niche — Derived from Diet Proportions

BirdBase provides quantitative diet proportions across 9 food categories (each out of 10).
We derive a `trophic_niche` label that matches AVONET's taxonomy by taking the
**dominant diet column**:
- Max proportion > 5 → that category's niche label
- Max proportion ≤ 5 → `Omnivore` (no clear dominant)
- All NaN / all zero → `MISSING`

In [ ]:
print("AVONET trophic_niche values:")
print(avonet["trophic_niche"].value_counts())

In [ ]:
DIET_PROPORTION_COLS = [
    'diet_invertebrate', 'diet_fruit', 'diet_nectar', 'diet_seed',
    'diet_vertebrate', 'diet_fish', 'diet_scavenge', 'diet_plant', 'diet_mushroom'
]

# Cast proportion columns to numeric (contain 'T' traces and MISSING strings)
for col in DIET_PROPORTION_COLS + ['diet_sum']:
    birdbase[col] = pd.to_numeric(
        birdbase[col].astype(str).str.replace(",", "").str.strip(),
        errors="coerce"
    )

birdbase["diet_invertebrate"].describe()

In [ ]:
DOMINANT_TO_NICHE = {
    'diet_invertebrate': 'Invertivore',
    'diet_fruit':        'Frugivore',
    'diet_nectar':       'Nectarivore',
    'diet_seed':         'Granivore',
    'diet_vertebrate':   'Vertivore',
    'diet_fish':         'Aquatic predator',
    'diet_scavenge':     'Scavenger',
    'diet_plant':        'Herbivore terrestrial',
    'diet_mushroom':     'Herbivore terrestrial',
}

def assign_trophic_niche(row):
    values = row[DIET_PROPORTION_COLS]
    if values.isna().all() or values.sum() == 0:
        return 'MISSING'
    max_val = values.max()
    if max_val == 0:
        return 'MISSING'
    elif max_val <= 5:
        return 'Omnivore'
    return DOMINANT_TO_NICHE[values.idxmax()]

birdbase['trophic_niche'] = birdbase.apply(assign_trophic_niche, axis=1)

print("BirdBase trophic_niche after derivation:")
print(birdbase['trophic_niche'].value_counts())

### 4.4 · Habitat — Vocabulary Harmonisation

BirdBase uses 14 habitat labels; AVONET uses 12.  We map BirdBase labels to AVONET's schema
so the `_BB` and main habitat columns are directly comparable.

In [ ]:
print("BirdBase habitat_primary:")
print(birdbase["habitat_primary"].value_counts())
print()
print("AVONET habitat:")
print(avonet["habitat"].value_counts())

In [ ]:
habitat_mapping = {
    'Forest':    'Forest',
    'Shrub':     'Shrubland',
    'Woodland':  'Woodland',
    'Grassland': 'Grassland',
    'Wetland':   'Wetland',
    'Coastal':   'Coastal',
    'Sea':       'Marine',
    'Savanna':   'Grassland',   # Savanna mapped to Grassland (open biome)
    'Riparian':  'Riverine',
    'Rocky':     'Rock',
    'Plains':    'Grassland',   # Plains mapped to Grassland (open biome)
    'Artificial':'Human Modified',
    'Desert':    'Desert',
    'Bamboo':    'Forest',      # Bamboo forests treated as Forest
}

birdbase['habitat_primary'] = birdbase['habitat_primary'].map(habitat_mapping).fillna('MISSING')

print("BirdBase habitat_primary after harmonisation:")
print(birdbase['habitat_primary'].value_counts())

### 4.5 · Migration Columns — Decision to Drop

BirdBase encodes migration behaviour across five binary columns
(`migratory`, `altitudinal_migrant`, `irregular_migrant`, `dispersive`, `sedentary`).
Missing-value analysis shows that most of these columns are extremely sparse:

In [ ]:
migration_cols = ['migratory', 'altitudinal_migrant', 'irregular_migrant', 'dispersive', 'sedentary']

for col in migration_cols:
    n_missing = (birdbase[col].astype(str) == 'MISSING').sum() + birdbase[col].isna().sum()
    pct = n_missing / len(birdbase) * 100
    print(f"{col:<25} {n_missing:>6,} missing  ({pct:.1f}%)")

**Conclusion:** These columns are too sparse (75–95 % missing) to be useful as features.
AVONET already provides a clean numeric `migration` score (1 = sedentary, 3 = migratory),
so we drop the BirdBase migration columns entirely.

### 4.6 · Elevation Columns — Missing-Value Audit

BirdBase records elevation using five columns:
`elev_min`, `elev_norm_min`, `elev_range`, `elev_norm_max`, `elev_max`.
We audit them to decide which to keep.

In [ ]:
elev_cols = ['elev_min', 'elev_norm_min', 'elev_range', 'elev_norm_max', 'elev_max']

missing_values_sentinel = ["nan", "NaN", "null", "NULL", "NA", "N/A", "MISSING", "", " "]
df_str = birdbase.astype(str)

for col in elev_cols:
    missing_mask = birdbase[col].isna() | df_str[col].isin(missing_values_sentinel)
    n = missing_mask.sum()
    pct = n / len(birdbase) * 100
    print(f"{col:<20} {n:>6,} missing  ({pct:.1f}%)")

**Conclusion:**
- `elev_min` (86 % missing) and `elev_max` (75 % missing) are too sparse to be useful — **dropped**.
- `elev_norm_min` and `elev_norm_max` are better covered and represent
  the normalised lower/upper bounds of a species' elevation range — **kept**.
- `elev_range` (6 % missing) is the most complete elevation feature — **kept**.

### 4.7 · Full Missing-Value Summary (BirdBase)

In [ ]:
df_str = birdbase.astype(str)
summary = {}
for col in birdbase.columns:
    missing_mask = birdbase[col].isna() | df_str[col].isin(missing_values_sentinel)
    summary[col] = missing_mask.sum()

pd.DataFrame.from_dict(summary, orient='index', columns=['missing_count']) \
  .sort_values('missing_count', ascending=False)

### 4.8 · Drop Sparse & Redundant Columns

Based on the missing-value audit above, we drop:
- **Mass range columns** (`mass_female_min/max`, `mass_male_min/max`, `mass_unsexed_min/max`) — `mass_avg` is already the usable summary
- **Raw elevation bounds** (`elev_min`, `elev_max`) — too sparse; normalised versions retained
- **Individual diet proportions** — we've already derived `trophic_niche` from them
- **Migration binary columns** — too sparse; AVONET already has `migration`
- **Taxonomy columns** (`order`, `family`, `genus`, `species_latin_key`) — duplicated in other sources

In [ ]:
DROP_COLS = [
    # Mass range (redundant given mass_avg)
    'mass_female_min', 'mass_female_max',
    'mass_male_min',   'mass_male_max',
    'mass_unsexed_min','mass_unsexed_max',
    # Elevation (too sparse)
    'elev_min', 'elev_max',
    # Diet proportions (trophic_niche already derived)
    'diet_invertebrate', 'diet_fruit', 'diet_nectar', 'diet_seed',
    'diet_vertebrate', 'diet_fish', 'diet_scavenge', 'diet_plant',
    'diet_mushroom', 'diet_sum',
    # Migration (too sparse)
    'migratory', 'altitudinal_migrant', 'irregular_migrant', 'dispersive', 'sedentary',
    # Taxonomy (available elsewhere)
    'order', 'family', 'genus', 'species_latin_key',
]

birdbase.drop(columns=DROP_COLS, inplace=True)

print(f"BirdBase columns remaining: {list(birdbase.columns)}")

## 5 · Column Renaming

Both datasets are renamed to clear, readable names before merging.
Columns that exist in **both** datasets (mass, habitat, trophic niche) get `_BB` suffixes
on the BirdBase side so both versions are preserved in the combined dataset.

In [ ]:
AVONET_RENAME = {
    # Beak measurements
    'beak_culmen_avg':   'beak_culmen',       'beak_culmen_avg_m': 'beak_culmen_m',      'beak_culmen_avg_f': 'beak_culmen_f',
    'beak_nares_avg':    'beak_nares',         'beak_nares_avg_m':  'beak_nares_m',       'beak_nares_avg_f':  'beak_nares_f',
    'beak_width_avg':    'beak_width',         'beak_width_avg_m':  'beak_width_m',       'beak_width_avg_f':  'beak_width_f',
    'beak_depth_avg':    'beak_depth',         'beak_depth_avg_m':  'beak_depth_m',       'beak_depth_avg_f':  'beak_depth_f',
    # Body measurements
    'tarsus_avg':        'tarsus',             'tarsus_avg_m':      'tarsus_m',           'tarsus_avg_f':      'tarsus_f',
    'wing_len_avg':      'wing_len',           'wing_len_avg_m':    'wing_len_m',         'wing_len_avg_f':    'wing_len_f',
    'kipps_avg':         'kipps',              'kipps_avg_m':       'kipps_m',            'kipps_avg_f':       'kipps_f',
    'secondary_avg':     'secondary',          'secondary_avg_m':   'secondary_m',        'secondary_avg_f':   'secondary_f',
    'hwi_avg':           'hand_wing_index',    'hwi_avg_m':         'hand_wing_index_m',  'hwi_avg_f':         'hand_wing_index_f',
    'tail_avg':          'tail_len',           'tail_avg_m':        'tail_len_m',         'tail_avg_f':        'tail_len_f',
    # Sample info
    'total_individuals': 'sample_size',
    'female_count':      'sample_size_f',
    'male_count':        'sample_size_m',
    # Ecology / taxonomy
    'mass_avg':          'mass',
    'mass_source':       'mass_source',
    'habitat':           'habitat',
    'habitat_density':   'habitat_density',
    'trophic_level':     'primary_diet',
    'trophic_niche':     'trophic_niche',
    # Geography
    'lat_centroid':      'lat_centroid',
    'lon_centroid':      'lon_centroid',
    'range_size':        'range_size',
    # unchanged: avibase_id, inference, migration, lifestyle, lat_min, lat_max
}

BIRDBASE_RENAME = {
    'mass_avg':        'mass_BB',           # conflict → _BB suffix
    'elev_norm_min':   'elevation_min_BB',
    'elev_range':      'elevation_range_BB',
    'elev_norm_max':   'elevation_max_BB',
    'habitat_primary': 'habitat_BB',        # conflict → _BB suffix
    'diet_primary':    'primary_diet_BB',
    'nest_type':       'nest_type_BB',
    'flightless':      'flightless_BB',
    'trophic_niche':   'trophic_niche_BB',  # conflict → _BB suffix
    # unchanged: avibase_id
}

avonet   = avonet.rename(columns=AVONET_RENAME)
birdbase = birdbase.rename(columns=BIRDBASE_RENAME)

print("AVONET columns   :", list(avonet.columns))
print()
print("BirdBase columns :", list(birdbase.columns))

## 6 · Resolve BirdBase Duplicates

BirdBase has 11,412 rows but only 11,345 unique `avibase_id` values — 67 species appear
more than once (different measurements from different source records).

**Resolution strategy:**
- **Numeric columns** → mean of all valid (non-null) values
- **Categorical columns** → first non-null value

In [ ]:
# Inspect duplicates
dupes = birdbase[birdbase.duplicated(subset='avibase_id', keep=False)]
print(f"Duplicate rows     : {len(dupes)}")
print(f"Duplicate IDs      : {dupes['avibase_id'].nunique()}")
print()
dupes.sort_values('avibase_id').head(10)

In [ ]:
# Normalise MISSING strings → NaN before resolving
birdbase = birdbase.replace('MISSING', np.nan)

BB_NUMERIC_COLS     = ['mass_BB', 'elevation_min_BB', 'elevation_range_BB', 'elevation_max_BB']
BB_CATEGORICAL_COLS = ['habitat_BB', 'primary_diet_BB', 'nest_type_BB', 'flightless_BB', 'trophic_niche_BB']

for col in BB_NUMERIC_COLS:
    birdbase[col] = pd.to_numeric(birdbase[col], errors='coerce')

def resolve_group(group):
    resolved = {}
    for col in BB_NUMERIC_COLS:
        valid = group[col].dropna()
        resolved[col] = valid.mean() if len(valid) > 0 else np.nan
    for col in BB_CATEGORICAL_COLS:
        valid = group[col].dropna()
        resolved[col] = valid.iloc[0] if len(valid) > 0 else np.nan
    resolved['avibase_id'] = group['avibase_id'].iloc[0]
    return pd.Series(resolved)

birdbase_clean = (
    birdbase
    .groupby('avibase_id', sort=False)
    .apply(resolve_group)
    .reset_index(drop=True)
)

print(f"Before dedup : {len(birdbase):,} rows")
print(f"After dedup  : {len(birdbase_clean):,} rows")
print(f"Unique IDs   : {birdbase_clean['avibase_id'].nunique():,}")

## 7 · Merge Datasets

We use an **outer join** on `avibase_id` so that every species present in either dataset
is retained.  Where a species exists in only one source, the other source's columns are `NaN`.

We also add two boolean provenance flags:
- `in_avonet` — True if the species has any non-null data from AVONET
- `in_birdbase` — True if the species has any non-null data from BirdBase

In [ ]:
combined = pd.merge(avonet, birdbase_clean, on='avibase_id', how='outer')

# Provenance flags
avonet_cols    = [c for c in avonet.columns    if c != 'avibase_id']
birdbase_cols  = [c for c in birdbase_clean.columns if c != 'avibase_id']

combined['in_avonet']   = combined[avonet_cols].notna().any(axis=1)
combined['in_birdbase'] = combined[birdbase_cols].notna().any(axis=1)

print(f"Combined shape : {combined.shape}")
print(f"Columns        : {list(combined.columns)}")

## 8 · Merge Summary

In [ ]:
total          = len(combined)
in_both        = (combined['in_avonet'] & combined['in_birdbase']).sum()
only_avonet    = (combined['in_avonet'] & ~combined['in_birdbase']).sum()
only_birdbase  = (~combined['in_avonet'] & combined['in_birdbase']).sum()

print(f"Total unique species  : {total:,}")
print(f"In both datasets      : {in_both:,}")
print(f"AVONET only           : {only_avonet:,}")
print(f"BirdBase only         : {only_birdbase:,}")

# Duplicate check
dupes = combined[combined.duplicated(subset='avibase_id', keep=False)]
if len(dupes):
    print(f"\nWarning: {len(dupes)} duplicate avibase_id rows found")
else:
    print("\nNo duplicate avibase_ids — merge is clean")

In [ ]:
combined.head()

## 9 · Save Combined Dataset

In [ ]:
combined.to_csv('merged_birdbase_avonet.csv', index=False)
print(f"Saved → combined_birds.csv  ({total:,} rows × {combined.shape[1]} cols)")

---
## Next Steps

- **Feature engineering** — log-transform mass and range_size; encode categorical columns
- **Missing-value imputation** — decide per-column strategy for downstream modelling
- **Conflict analysis** — compare `mass` vs `mass_BB`, `habitat` vs `habitat_BB`, `trophic_niche` vs `trophic_niche_BB` to assess inter-source agreement